# LegalIR Task 1 — Colab A100, PyVi speedup release candidate
Pinned Git Commit: `d53ffc342b64937e41e8daa08ac8ce6f19e0a0ab`

Apply and commit the source patch, complete the freeze/CI/T4 gate rollover described
in the supplied PDFs, then regenerate this notebook. The historical pin does not
contain this fix; Cell 2 deliberately detects that before dataset/model downloads.

Training, candidate budgets and score math are unchanged. Under one hour is a target,
not a verified result. Use a fresh Colab A100 runtime, run cells in order, and set
DATASET_DIR to your already released dataset (do not rebuild the release here).
The optional benchmark reports the PyVi stage separately from full training.


In [ ]:
# Cell 1: A100 and local settings
import os, sys, json, time, subprocess
from pathlib import Path
NOTEBOOK_STARTED = time.perf_counter()
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("LEGALIR_PYVI_MAX_WORKERS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
DATASET_DIR = os.environ.get("LEGALIR_DATASET_DIR", "/content/kaggle_dataset")
OUTPUT_DIR = os.environ.get("LEGALIR_OUTPUT_DIR", "/content/legalir_production_run")
# Set OUTPUT_DIR to a persistent directory if cache must survive VM deletion.
import torch
assert torch.cuda.is_available(), "Select an A100 runtime in Colab first."
assert "A100" in torch.cuda.get_device_name(0), "This notebook requires an A100."
assert torch.cuda.is_bf16_supported(), "Native BF16 support is required."
print({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "python": sys.version.split()[0]})
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is not None:
    for key in ("HF_TOKEN", "HF_TOKEN_READ", "HF_TOKEN_WRITE", "HF_REPO_ID"):
        try:
            value = userdata.get(key)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            continue
        if value:
            os.environ.setdefault(key, str(value))
# Tokens remain only in the environment and are never printed.

In [ ]:
# Cell 2: exact source checkout and feature check before expensive work
import re
EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA", "d53ffc342b64937e41e8daa08ac8ce6f19e0a0ab")
assert re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_COMMIT), "Use a full approved Git SHA."
REPO_DIR = Path(os.environ.get("LEGALIR_REPO_DIR", "/content/LegalIR"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)
assert (REPO_DIR / ".git").exists(), "REPO_DIR must be a Git checkout."
dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True)
assert not dirty.strip(), "Commit/review local source edits before selecting the approved release SHA."
subprocess.run(["git", "fetch", "origin", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
assert actual.lower() == EXPECTED_COMMIT.lower()
source = (REPO_DIR / "src/retrieval/bm25_pyvi.py").read_text(encoding="utf-8")
assert 'PYVI_INDEX_CODE_VERSION = "bm25_pyvi_v3"' in source, (
    "BLOCKED: this approved pin predates the PyVi fix. Apply the patch, complete "
    "the freeze/CI/T4 rollover and regenerate notebooks; do not bypass gates."
)
print("Source verified:", actual)

In [ ]:
# Cell 3: pinned tested Python stack, retain Colab's CUDA PyTorch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements-colab-a100.txt")], check=True)
# Probe in a fresh interpreter so packages imported in previous cells cannot be stale.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, pyvi, pyarrow; print('Dependency imports PASS')"], check=True)
dataset_dir = Path(DATASET_DIR).expanduser().resolve()
required = ("documents.parquet", "chunks.parquet", "queries_train.parquet", "qrels_train.parquet", "public-official.json")
if (dataset_dir / "canonical").is_dir():
    dataset_dir = dataset_dir / "canonical"
missing = [name for name in required if not (dataset_dir / name).is_file()]
assert not missing, f"Dataset incomplete at {dataset_dir}: {missing}. Point DATASET_DIR at the existing release."
print("Dataset files found:", dataset_dir)
# Full checksum/schema/freeze validation still runs in the authoritative gate.

In [ ]:
# Cell 4: optional >=20k real-data PyVi benchmark; does NOT run training
RUN_PYVI_BENCHMARK = False
if RUN_PYVI_BENCHMARK:
    subprocess.run([sys.executable, "-u", "scripts/benchmark_pyvi_build.py",
                    "--dataset-dir", str(dataset_dir), "--rows", "20000",
                    "--workers", os.environ["LEGALIR_PYVI_MAX_WORKERS"],
                    "--output", str(Path(OUTPUT_DIR) / "pyvi_benchmark.json")], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 5: authoritative production process; stream stdout and stderr immediately
# scripts/run_colab_train.py -> scripts/gates/run_a100.py
output_dir = Path(OUTPUT_DIR).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, "-u", "scripts/run_colab_train.py",
           "--dataset-dir", str(dataset_dir), "--output-dir", str(output_dir),
           "--expected-sha", EXPECTED_COMMIT, "--precision", "bf16", "--mode", "full"]
# No artificial 1-hour kill: stopping early would not be completed training.
subprocess.run(command, cwd=REPO_DIR, check=True)
report = json.loads((output_dir / "run_manifest.json").read_text(encoding="utf-8"))
assert report.get("verdict") == "PASS" and not report.get("mock", False), report.get("status")
elapsed = time.perf_counter() - NOTEBOOK_STARTED
summary = {"elapsed_seconds": elapsed, "target_seconds": 3600,
           "under_one_hour": elapsed < 3600, "status": report.get("status"),
           "scope": "cells1-through-production-including-optional-benchmark-and-user-pauses"}
(output_dir / "notebook_runtime.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(summary)
print("Submission:", output_dir / "submission.zip")